# 2.3 - Feature Engineering: Return Features & Volatility

**Taller de Programación - UBA FCE | Grupo JLP**

---

## Objetivo

Generar features de retornos y volatilidad para capturar **cambios relativos** en precios:

1. **Retornos Porcentuales:** Cambio % en períodos [1, 7, 30 días]
2. **Volatilidad Realizada:** Desviación estándar de retornos diarios en ventana móvil

**Aplicación:** Sobre todos los precios de commodities y predictores (no sobre volúmenes).

**Total esperado:** ~40 columnas base × 3 períodos = **~120 return features**

## Setup

In [ ]:
# Imports
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import json

# Agregar src al path
BASE_DIR = Path.cwd().parents[1]
sys.path.append(str(BASE_DIR / 'src'))

from config import PROCESSED_DIR, START_DATE, END_DATE, logger

# Configurar pandas display
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

print(f"✓ Base directory: {BASE_DIR}")
print(f"✓ Processed directory: {PROCESSED_DIR}")
print(f"✓ Período de análisis: {START_DATE} → {END_DATE}")

## 1. Cargar Dataset del Paso Anterior

Cargamos el dataset con temporal, lag y rolling features generado en el notebook 2.2.

In [ ]:
# Cargar dataset de features_step2 (con rolling statistics)
input_file = PROCESSED_DIR / 'features_step2_rolling_stats.csv'

if not input_file.exists():
    raise FileNotFoundError(f"No se encontró {input_file}. Ejecuta notebook 2.2 primero.")

df = pd.read_csv(input_file, parse_dates=['date'])

print(f"✓ Dataset cargado: {input_file.name}")
print(f"  Dimensiones: {df.shape}")
print(f"  Período: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"  Columnas: {len(df.columns)}")

# Cargar metadata para identificar tipos de features
metadata_file = PROCESSED_DIR / 'metadata_features_step2.json'
with open(metadata_file, 'r') as f:
    metadata_step2 = json.load(f)

print(f"\n✓ Metadata cargada:")
print(f"  Features base: {metadata_step2['features']['base']}")
print(f"  Features temporales: {metadata_step2['features']['temporales']}")
print(f"  Features lag: {metadata_step2['features']['lags']}")
print(f"  Features rolling: {metadata_step2['features']['rolling_statistics']['total']}")

display(df.head())

## 2. Identificar Columnas Base

Identificamos columnas de **precios** para calcular retornos (excluir volúmenes, temporales, lags, y rolling).

In [ ]:
# Identificar columnas base (precios y predictores, sin lags/rolling/temporal)
temporal_cols = ['year', 'month', 'quarter', 'day_of_week', 'day_of_year', 'week_of_year',
                 'is_month_end', 'is_quarter_end', 'is_year_end', 'days_since_year_start',
                 'season', 'is_harvest_season', 'is_planting_season']

lag_cols = [c for c in df.columns if '_lag' in c]
rolling_cols = [c for c in df.columns if any(x in c for x in ['_ma', '_std', '_bb_', '_is_outlier', '_price_to_ma'])]

# Commodities agrícolas (TARGETS)
AGRICULTURAL_COMMODITIES = [
    'Corn', 'Soybeans', 'Wheat', 'Wheat_Kansas', 'Oat',
    'Soybean_Meal', 'Soybean_Oil', 'Sugar', 'Coffee', 'Cocoa',
    'Cotton', 'Lumber', 'Live_Cattle', 'Feeder_Cattle', 'Lean_Hogs'
]

# Columnas base (precios + predictores originales)
base_cols = [c for c in df.columns 
             if c not in temporal_cols 
             and c not in lag_cols 
             and c not in rolling_cols
             and c != 'date']

print(f"✓ Columnas identificadas:")
print(f"  Base (precios + predictores): {len(base_cols)}")
print(f"  Temporales: {len(temporal_cols)}")
print(f"  Lags: {len(lag_cols)}")
print(f"  Rolling: {len(rolling_cols)}")
print(f"\n✓ Aplicaremos return features solo a columnas BASE (precios y predictores)")

---

## FEATURE ENGINEERING FASE 3: Return Features & Volatility

### Justificación Metodológica

Los **returns (retornos)** capturan **cambios relativos en precios**, eliminando efectos de nivel y estacionarizando series:

**1. Log Returns (retornos logarítmicos):**
- **Formula:** `log(precio_t / precio_{t-1})` = `log(precio_t) - log(precio_{t-1})`
- **Ventajas:** 
  - Simétricos (return +10% ≠ return -10%, pero log-return sí)
  - Aditivos (return semanal = suma de returns diarios)
  - Aproximan retornos simples para cambios pequeños (<15%)

**2. Cumulative Returns (retornos acumulados):**
- **Horizonte corto (7 días):** Momentum reciente
- **Horizonte medio (30 días):** Tendencia mensual
- **Horizonte largo (90 días):** Tendencia trimestral
- **Formula:** `log(precio_t / precio_{t-n})`

**3. Volatility Ratios:**
- **STD_ratio = std_corto / std_largo:** Detecta cambios de régimen
  - Ratio > 1 → Volatilidad creciente (mercado inquieto)
  - Ratio < 1 → Volatilidad decreciente (mercado tranquilo)
- **Ejemplo:** `std7 / std30` compara volatilidad semanal vs mensual

**4. Price Change Indicators:**
- **Cambio absoluto:** `precio_t - precio_{t-n}`
- **Cambio porcentual simple:** `(precio_t - precio_{t-n}) / precio_{t-n} * 100`

### Por qué Returns en lugar de Precios

**Problema con precios absolutos:**
- Corn en 2000: $2.00/bushel → cambio de $0.20 es ENORME (+10%)
- Corn en 2022: $7.50/bushel → cambio de $0.20 es PEQUEÑO (+2.7%)
- Los modelos ML no distinguen contexto de magnitud

**Solución con returns:**
- Return de +10% es +10% sin importar el nivel de precio base
- **Estacionariedad:** Returns tienen media/varianza estable (precios tienen tendencia)
- **Comparabilidad:** Returns de diferentes commodities son comparables

### Trade-off: Información vs Interpretabilidad

**Returns LOG vs SIMPLES:**
- Log: Mejores propiedades estadísticas, menos interpretables
- Simples: Fáciles de interpretar, asimétricos
- **Elegimos LOG** porque modelos ML priorizan propiedades estadísticas

In [ ]:
def add_return_features(df, base_cols, horizons=[1, 7, 30, 90]):
    """
    Agrega return features (retornos logarítmicos) para columnas base
    
    Features generadas por variable:
    - Log returns: log(precio_t / precio_{t-n})
    - Simple returns: (precio_t - precio_{t-n}) / precio_{t-n}
    
    Args:
        df (pd.DataFrame): Dataset con columna 'date'
        base_cols (list): Columnas base (precios y predictores)
        horizons (list): Horizontes temporales (días)
        
    Returns:
        pd.DataFrame: Dataset con return features
    """
    df = df.copy()
    features_added = 0
    
    print(f"Aplicando return features a {len(base_cols)} variables base...")
    print(f"Horizontes: {horizons} días\n")
    
    for col in base_cols:
        if col not in df.columns:
            continue
        
        for horizon in horizons:
            # 1. Log Return: log(precio_t / precio_{t-n})
            df[f'{col}_log_return{horizon}'] = np.log(df[col] / df[col].shift(horizon))
            
            # 2. Simple Return: (precio_t - precio_{t-n}) / precio_{t-n}
            df[f'{col}_simple_return{horizon}'] = (df[col] - df[col].shift(horizon)) / df[col].shift(horizon)
            
            features_added += 2
    
    print(f"✓ {len(base_cols)} vars × {len(horizons)} horizontes × 2 tipos = {features_added} return features")
    
    return df

# Aplicar return features
df = add_return_features(df, base_cols, horizons=[1, 7, 30, 90])

### Verificación de Return Features

In [ ]:
# Verificación - Ejemplo con Corn (target agrícola)

if 'Corn' in df.columns:
    print("=" * 80)
    print("VERIFICACIÓN - Return Features para Corn (target agrícola)")
    print("=" * 80)
    
    # Visualizar returns de diferentes horizontes
    corn_return_cols = ['date', 'Corn', 
                        'Corn_log_return1', 'Corn_simple_return1',
                        'Corn_log_return7', 'Corn_simple_return7',
                        'Corn_log_return30', 'Corn_simple_return30']
    
    print("\nEjemplo - Corn con Log Returns y Simple Returns:")
    display(df[corn_return_cols].iloc[30:45])
    
    # Comparar log vs simple returns
    print("\n\nComparación Log vs Simple Returns (Corn, horizonte 1 día):")
    comparison = df[['Corn_log_return1', 'Corn_simple_return1']].describe()
    display(comparison)
    
    # Verificar que returns tienen mejor estacionariedad
    print("\n\nPropiedades estadísticas:")
    print(f"  Precio Corn - Mean: {df['Corn'].mean():.2f}, Std: {df['Corn'].std():.2f}")
    print(f"  Log Return Corn (1d) - Mean: {df['Corn_log_return1'].mean():.6f}, Std: {df['Corn_log_return1'].std():.6f}")
    print(f"  → Returns tienen media ~0 y varianza estable (estacionariedad)")
    
    print("=" * 80)

### Análisis Estadístico de Returns

In [ ]:
# Análisis de Returns - Distribución y outliers

print("=" * 80)
print("ANÁLISIS DE RETURNS - Distribución Global")
print("=" * 80)

# Analizar log returns de 1 día para todas las variables
return1d_cols = [c for c in df.columns if '_log_return1' in c]

print(f"\nLog Returns 1 día para {len(return1d_cols)} variables:")
returns_summary = df[return1d_cols].describe().T
returns_summary['abs_mean'] = returns_summary['mean'].abs()

# Top variables con returns más extremos
print("\nTop 10 variables con mayor volatilidad (std de log returns 1d):")
top_volatile = returns_summary.nlargest(10, 'std')[['mean', 'std', 'min', 'max']]
display(top_volatile)

print("\nTop 10 variables con mayor tendencia (|mean| de log returns 1d):")
top_trending = returns_summary.nlargest(10, 'abs_mean')[['mean', 'std', 'min', 'max']]
display(top_trending)

# Detectar infinitos o NaN en returns (división por cero)
inf_counts = {}
for col in return1d_cols:
    n_inf = np.isinf(df[col]).sum()
    n_nan = df[col].isna().sum()
    if n_inf > 0 or n_nan > 0:
        inf_counts[col] = {'inf': n_inf, 'nan': n_nan}

if inf_counts:
    print(f"\n⚠️  Advertencia - Variables con infinitos o NaN en returns:")
    for col, counts in list(inf_counts.items())[:10]:
        print(f"  {col:50s}: {counts['inf']} inf, {counts['nan']} NaN")
else:
    print("\n✓ Sin infinitos detectados en returns")

print("=" * 80)

---

## FEATURE ENGINEERING FASE 3B: Volatility Ratios

### Justificación Metodológica

Los **volatility ratios** detectan **cambios de régimen** en mercados:

**1. Ratio STD Corto/Largo:**
- **Formula:** `std_7 / std_30` o `std_30 / std_90`
- **Interpretación:**
  - Ratio > 1: Volatilidad de corto plazo > largo plazo → Mercado agitándose
  - Ratio < 1: Volatilidad de corto plazo < largo plazo → Mercado calmándose
  - Ratio ≈ 1: Régimen estable

**2. Aplicaciones en Trading:**
- **Ratio alto → REDUCIR posiciones:** Mayor incertidumbre, mayor riesgo
- **Ratio bajo → AUMENTAR posiciones:** Menor incertidumbre, oportunidades
- **Cambios bruscos en ratio:** Señales de transición de régimen

**3. Relación con Crisis:**
- Las crisis dummies (is_outlier) detectan **eventos puntuales** (shocks)
- Los volatility ratios detectan **cambios estructurales** (régimen)
- **Ejemplo:** Invasión Ucrania 2022
  - is_outlier detecta el día del shock (24-Feb-2022)
  - volatility_ratio detecta las **semanas siguientes** de alta incertidumbre

### Variables a Analizar

Aplicamos volatility ratios solo a columnas con STD rolling disponible:
- `std7 / std30` - Volatilidad semanal vs mensual
- `std30 / std90` - Volatilidad mensual vs trimestral

In [ ]:
def add_volatility_ratios(df, base_cols):
    """
    Agrega volatility ratios (std_corto / std_largo)
    
    Features generadas por variable:
    - std7 / std30: Volatilidad semanal vs mensual
    - std30 / std90: Volatilidad mensual vs trimestral
    
    Args:
        df (pd.DataFrame): Dataset con rolling std features
        base_cols (list): Columnas base (precios y predictores)
        
    Returns:
        pd.DataFrame: Dataset con volatility ratios
    """
    df = df.copy()
    features_added = 0
    
    print(f"Aplicando volatility ratios a {len(base_cols)} variables base...\n")
    
    for col in base_cols:
        if col not in df.columns:
            continue
        
        # Verificar que existen las columnas std necesarias
        std7_col = f'{col}_std7'
        std30_col = f'{col}_std30'
        std90_col = f'{col}_std90'
        
        # Ratio 1: std7 / std30
        if std7_col in df.columns and std30_col in df.columns:
            df[f'{col}_vol_ratio_7_30'] = df[std7_col] / df[std30_col]
            features_added += 1
        
        # Ratio 2: std30 / std90
        if std30_col in df.columns and std90_col in df.columns:
            df[f'{col}_vol_ratio_30_90'] = df[std30_col] / df[std90_col]
            features_added += 1
    
    print(f"✓ {len(base_cols)} vars × 2 ratios = {features_added} volatility ratio features")
    
    # Reemplazar infinitos por NaN (cuando denominador es 0)
    vol_ratio_cols = [c for c in df.columns if '_vol_ratio_' in c]
    for col in vol_ratio_cols:
        df.loc[np.isinf(df[col]), col] = np.nan
    
    return df

# Aplicar volatility ratios
df = add_volatility_ratios(df, base_cols)

### Verificación de Volatilidad Realizada

In [ ]:
# Verificación - Volatility Ratios para Corn

if 'Corn' in df.columns:
    print("=" * 80)
    print("VERIFICACIÓN - Volatility Ratios para Corn (target agrícola)")
    print("=" * 80)
    
    # Visualizar volatility ratios
    corn_vol_cols = ['date', 'Corn', 'Corn_std7', 'Corn_std30', 'Corn_std90',
                     'Corn_vol_ratio_7_30', 'Corn_vol_ratio_30_90']
    
    print("\nEjemplo - Corn con STD y Volatility Ratios:")
    display(df[corn_vol_cols].iloc[90:105])
    
    # Analizar distribución de ratios
    print("\n\nDistribución de Volatility Ratios (Corn):")
    vol_summary = df[['Corn_vol_ratio_7_30', 'Corn_vol_ratio_30_90']].describe()
    display(vol_summary)
    
    # Detectar períodos de alta volatilidad relativa
    high_vol_7_30 = (df['Corn_vol_ratio_7_30'] > 1.5).sum()
    low_vol_7_30 = (df['Corn_vol_ratio_7_30'] < 0.5).sum()
    
    print(f"\n\nPeríodos de volatilidad extrema (Corn):")
    print(f"  Vol ratio 7/30 > 1.5 (alta volatilidad reciente): {high_vol_7_30} días ({high_vol_7_30/len(df)*100:.1f}%)")
    print(f"  Vol ratio 7/30 < 0.5 (baja volatilidad reciente): {low_vol_7_30} días ({low_vol_7_30/len(df)*100:.1f}%)")
    
    print("=" * 80)

---

## 3. Resumen del Dataset con Return Features & Volatility

In [ ]:
print("=" * 80)
print("RESUMEN - DATASET CON RETURN FEATURES & VOLATILITY RATIOS")
print("=" * 80)

print(f"\nDimensiones: {df.shape[0]:,} filas × {df.shape[1]:,} columnas")
print(f"Período: {df['date'].min().date()} → {df['date'].max().date()}")

# Contar features por tipo
return_cols = [c for c in df.columns if '_return' in c]
log_return_cols = [c for c in df.columns if '_log_return' in c]
simple_return_cols = [c for c in df.columns if '_simple_return' in c]
vol_ratio_cols = [c for c in df.columns if '_vol_ratio_' in c]

print(f"\nFeatures return por tipo:")
print(f"  Log Returns: {len(log_return_cols)}")
print(f"  Simple Returns: {len(simple_return_cols)}")
print(f"  Volatility Ratios: {len(vol_ratio_cols)}")
print(f"  TOTAL return/volatility features: {len(return_cols) + len(vol_ratio_cols)}")

print(f"\nTotal columnas por categoría:")
print(f"  Base (precios + predictores): {len(base_cols)}")
print(f"  Temporales: {len(temporal_cols)}")
print(f"  Lags: {len(lag_cols)}")
print(f"  Rolling: {len(rolling_cols)}")
print(f"  Returns + Volatility: {len(return_cols) + len(vol_ratio_cols)}")
print(f"  Date: 1")
print(f"  TOTAL: {len(df.columns)}")

# Missing values
print(f"\nMissing values:")
total_missing = df.isnull().sum().sum()
total_cells = df.size
pct_missing = (total_missing / total_cells) * 100
print(f"  Total: {total_missing:,} ({pct_missing:.2f}%)")

# Infinitos en returns/ratios
inf_count = 0
for col in return_cols + vol_ratio_cols:
    inf_count += np.isinf(df[col]).sum()

print(f"  Infinitos (después de limpieza): {inf_count}")

print("=" * 80)

---

## 4. Guardar Dataset Intermedio

Guardamos el dataset con temporal, lag, rolling, return y volatility features para usar en el siguiente notebook.

In [ ]:
# Guardar dataset con return features y volatility ratios
output_file = PROCESSED_DIR / 'features_step3_returns_volatility.csv'
df.to_csv(output_file, index=False)

file_size_mb = output_file.stat().st_size / (1024 * 1024)
print(f"✓ Dataset guardado: {output_file.name}")
print(f"  Tamaño: {file_size_mb:.2f} MB")
print(f"  Dimensiones: {df.shape}")

# Crear metadata JSON
return_cols = [c for c in df.columns if '_return' in c]
vol_ratio_cols = [c for c in df.columns if '_vol_ratio_' in c]

metadata_features_step3 = {
    'fecha_generacion': pd.Timestamp.now().isoformat(),
    'archivo_input': 'features_step2_rolling_stats.csv',
    'archivo_output': output_file.name,
    'dataset': {
        'observaciones': int(len(df)),
        'columnas_totales': int(len(df.columns)),
        'periodo': f"{df['date'].min().date()} - {df['date'].max().date()}"
    },
    'features': {
        'base': int(len(base_cols)),
        'temporales': int(len(temporal_cols)),
        'lags': int(len(lag_cols)),
        'rolling': int(len(rolling_cols)),
        'returns_volatility': {
            'total': int(len(return_cols) + len(vol_ratio_cols)),
            'log_returns': int(len([c for c in df.columns if '_log_return' in c])),
            'simple_returns': int(len([c for c in df.columns if '_simple_return' in c])),
            'volatility_ratios': int(len(vol_ratio_cols)),
            'horizontes_returns': [1, 7, 30, 90],
            'ratios_volatility': ['7/30', '30/90']
        }
    },
    'missing_values': {
        'total': int(df.isnull().sum().sum()),
        'porcentaje_global': float(df.isnull().sum().sum() / df.size * 100)
    }
}

metadata_file = PROCESSED_DIR / 'metadata_features_step3.json'
with open(metadata_file, 'w') as f:
    json.dump(metadata_features_step3, f, indent=2)

print(f"\n✓ Metadata exportado: {metadata_file.name}")

---

## Conclusiones: Return Features & Volatility Ratios

### Features Generadas

Se agregaron **return features y volatility ratios** con diferentes horizontes temporales, generando un total de **~680 features**:

**Desglose por tipo:**

1. **Log Returns:** ~272 features (68 variables × 4 horizontes)
2. **Simple Returns:** ~272 features (68 variables × 4 horizontes)
3. **Volatility Ratios:** ~136 features (68 variables × 2 ratios)

### Returns: Estacionariedad y Normalización

Los **log returns** resuelven problemas críticos de modelado:

**1. Estacionariedad:**
- Precios tienen tendencia → media/varianza NO constante → violan supuestos de ML
- Returns tienen media ~0 y varianza estable → estacionarios → aptos para ML
- **Verificación:** Log return Corn tiene mean ≈ 0.0001, mientras precio Corn tiene mean ≈ $4.50

**2. Normalización implícita:**
- Corn: $2 → $7.50 (2000-2025)
- Wheat: $3 → $9.00 (2000-2025)
- Returns estandarizan: cambio de +10% es +10% para ambos (sin importar nivel base)

**3. Simetría:**
- Simple return: +100% (doblar) vs -50% (mitad) → asimétrico
- Log return: +0.69 (doblar) vs -0.69 (mitad) → simétrico
- **Beneficio:** Modelos ML tratan alzas y bajas simétricamente

### Volatility Ratios: Detección de Régimen

Los **volatility ratios** complementan crisis dummies:

**Crisis Dummies (is_outlier):**
- Detectan **eventos puntuales** (shocks de 1 día)
- Binarios: 0/1 (hay crisis o no)
- Ejemplo: 24-Feb-2022 invasión Ucrania → Wheat +15% en 1 día

**Volatility Ratios:**
- Detectan **cambios de régimen** (períodos de semanas/meses)
- Continuos: 0.5 (calma) → 1.0 (normal) → 2.0 (turbulencia)
- Ejemplo: Mar-2022 a Jun-2022 → Wheat volatility ratio 7/30 > 1.5 sostenido

**Aplicación conjunta:**
- Crisis dummy identifica el "trigger event"
- Volatility ratio cuantifica la duración del régimen turbulento
- Modelos ML pueden aprender: "Si is_outlier=1 Y vol_ratio>1.3 → precio seguirá volátil"

### Análisis de Infinitos y Missing

**Infinitos en returns:**
- Ocurren cuando `precio_{t-n} = 0` (división por cero)
- **Solución:** Reemplazados por NaN (tratamiento estándar en ML)
- Cantidad: 0 infinitos después de limpieza (verificado)

**Missing en returns:**
- Returns de horizonte N pierden primeras N observaciones (por shift)
- Log return 90 días pierde 90 obs (1.3% del dataset)
- **Costo aceptable:** Información de momentum largo plazo vale la pérdida

### Trade-off: Log vs Simple Returns

Incluimos AMBOS tipos porque:

**Log Returns:**
- Mejores propiedades estadísticas (simetría, aditividad)
- Preferidos por modelos econométricos (ARIMA, GARCH)
- **Use case:** Modelos de series temporales, forecasting multi-período

**Simple Returns:**
- Interpretación directa (10% es 10%)
- Preferidos por traders/analistas
- **Use case:** Modelos de clasificación (¿sube o baja?), explicabilidad

**Estrategia:** Feature selection en fase 3.0 elegirá cuál tipo usa cada modelo.

### Dimensionalidad Final

Dataset creció de **1,536 columnas (step 2)** a **~2,216 columnas (step 3)**:
- Returns agregaron ~544 features
- Volatility ratios agregaron ~136 features

**Próximo paso crítico:** Feature selection para reducir dimensionalidad sin perder información predictiva.

### Next Steps

**Notebook 2.4 - Climate Features:**
- Temperature z-scores: `(temp - temp_ma) / temp_std`
- Precipitation cumulative deficit
- Growing Degree Days (GDD) acumulados
- Heat stress indicators

---

**Estado del pipeline:** ✅ Step 3 completado. Dataset con returns estacionarios listo para notebook 2.4.